# Layer-Wise LoRA Retrieval & Hybrid Composition
### Extension of LoraRetriever (arXiv 2402.09997)

---

## 🧠 Core Idea: Why Layer-Wise?

The original LoraRetriever assigns **one embedding per task** (mean of all layers) and retrieves whole LoRAs. But this ignores a fundamental property of transformers:

> **Different layers specialise in different things.** Early layers capture syntax and positional structure. Middle layers handle semantic composition. Later layers encode task-specific reasoning and output formatting.

So for a new unseen task that combines two skills — say, reading comprehension + sentiment classification — the *ideal adapter* might borrow early layers from a reading comprehension LoRA and later layers from a sentiment LoRA. Forcing a single whole-LoRA retrieval cannot express this.

---

## 📐 What Changes from the Original

```
ORIGINAL (task-level):
  LoRA index:   { task_A → emb_A,  task_B → emb_B, ... }    (1 emb per task)
  Retrieval:    query → top-k tasks
  Composition:  merge top-k WHOLE LoRAs

THIS NOTEBOOK (layer-level):
  LoRA index:   { task_A → { layer_0: emb,  layer_1: emb, ... }
                  task_B → { layer_0: emb,  layer_1: emb, ... } }
  Retrieval:    query → for EACH LAYER independently → best task for that layer
  Composition:  build a HYBRID LoRA where each layer's weights come from a different task
```

---

## 🔬 How We Embed a LoRA Layer

The challenge: how do we get a meaningful embedding for **layer L of task T's LoRA**?

We use **activation-delta embeddings**:

```
For each few-shot example x of task T:
  1. Run base model forward pass with hooks → capture hidden state h_L (input to layer L)
  2. Compute LoRA's contribution at layer L:
       Δh_L  =  (h_L @ A_L^T) @ B_L^T  *  scaling
              where A_L ∈ ℝ^{r×d}, B_L ∈ ℝ^{d×r}
  3. Mean-pool Δh_L over the token dimension  →  vector ∈ ℝ^d
  4. L2-normalise

Mean-pool across all examples  →  final layer embedding ∈ ℝ^d
```

This embedding directly answers: *"what transformation does this LoRA layer apply to typical activations?"*  
Two LoRA layers that produce similar activation deltas will have similar embeddings — regardless of whether their raw A/B matrices look similar.

**Why not just SVD the ΔW matrix?**  
SVD of ΔW captures the geometric structure of the weight perturbation but ignores the *input distribution*. Two LoRAs could have identical singular values but act very differently on the actual inputs your model sees. Activation-deltas are input-aware.

---

## 🏗️ Hybrid LoRA Composition

After per-layer retrieval, we have an assignment like:

```
layer 0  q_proj  →  task: sentiment_sst2
layer 0  v_proj  →  task: nli_mnli
layer 1  q_proj  →  task: qa_squad
...
layer 31 q_proj  →  task: summarization_xsum
```

We then materialise a single `adapter_model.safetensors` from these per-layer selections. This is a valid PEFT adapter that can be loaded normally.

---

## 📦 0. Setup

In [ ]:
# Assumes LoraRetriever repo is already cloned and base environment is set up.
# If not, run the setup from LoraRetriever_Reproduction.ipynb first.

import os, json, random, copy, pickle
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Tuple, Optional

import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import pandas as pd

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
if device == 'cuda':
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# ---- Paths (adjust to your setup) ----
LORA_DIR      = Path("./lora_modules")      # downloaded LoRA adapters
DATA_DIR      = Path("./dataset")            # few-shot examples per task
INDEX_CACHE   = Path("./layer_wise_index")   # where we'll save the new index
INDEX_CACHE.mkdir(exist_ok=True)

## 🦙 1. Load Base Model (Once, Frozen)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, PeftConfig

BASE_MODEL_ID = "meta-llama/Llama-2-7b-hf"
N_SAMPLES     = 20    # few-shot examples per task used for embedding
MAX_SEQ_LEN   = 256   # truncate to keep memory manageable

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading base model (frozen)...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
base_model.eval()
for p in base_model.parameters():
    p.requires_grad = False

print("Base model ready.")

# ---- Discover LoRA target modules and layer names ----
# For Llama-2, LoRA is typically applied to q_proj, v_proj (and sometimes k_proj, o_proj)
# We enumerate them by peeking at the first available LoRA config
sample_lora_dir = next(LORA_DIR.iterdir())
peft_config     = PeftConfig.from_pretrained(str(sample_lora_dir))
TARGET_MODULES  = list(peft_config.target_modules)  # e.g. ['q_proj', 'v_proj']
LORA_R          = peft_config.r
LORA_ALPHA      = peft_config.lora_alpha
LORA_SCALING    = LORA_ALPHA / LORA_R

print(f"Target modules : {TARGET_MODULES}")
print(f"LoRA rank      : {LORA_R}")
print(f"Scaling factor : {LORA_SCALING}")

# Build the canonical list of all LoRA layer keys
# Format: "model.layers.{L}.self_attn.{module}" for L in 0..31
N_LAYERS = base_model.config.num_hidden_layers  # 32 for Llama-2-7b
LORA_LAYER_KEYS = [
    f"model.layers.{L}.self_attn.{mod}"
    for L in range(N_LAYERS)
    for mod in TARGET_MODULES
]
print(f"Total LoRA layers: {len(LORA_LAYER_KEYS)}  ({N_LAYERS} transformer layers × {len(TARGET_MODULES)} modules)")

## 🔑 2. Load LoRA Weights into a Structured Dict

We load every task's A and B matrices for every layer into a clean data structure:
```
lora_weights_db[task_name][layer_key] = {"A": tensor(r, d_in), "B": tensor(d_out, r)}
```

In [ ]:
import safetensors.torch as safetensors_utils

def load_lora_ab(lora_dir: Path) -> Dict[str, Dict[str, torch.Tensor]]:
    """
    Load all A and B matrices from a LoRA adapter directory.
    Returns: { layer_key: {"A": tensor, "B": tensor} }
    where layer_key = "model.layers.{L}.self_attn.{mod}"
    """
    sf_file  = lora_dir / "adapter_model.safetensors"
    bin_file = lora_dir / "adapter_model.bin"
    
    if sf_file.exists():
        state_dict = safetensors_utils.load_file(str(sf_file))
    elif bin_file.exists():
        state_dict = torch.load(str(bin_file), map_location="cpu")
    else:
        raise FileNotFoundError(f"No adapter weights in {lora_dir}")
    
    layers = defaultdict(dict)
    for raw_key, tensor in state_dict.items():
        # raw_key example: "base_model.model.model.layers.0.self_attn.q_proj.lora_A.weight"
        if "lora_A.weight" in raw_key:
            # Normalise to canonical key
            layer_key = (raw_key
                         .replace("base_model.model.", "")
                         .replace(".lora_A.weight", ""))
            layers[layer_key]["A"] = tensor.float()  # (r, d_in)
        elif "lora_B.weight" in raw_key:
            layer_key = (raw_key
                         .replace("base_model.model.", "")
                         .replace(".lora_B.weight", ""))
            layers[layer_key]["B"] = tensor.float()  # (d_out, r)
    
    # Keep only complete (A+B) layers
    complete = {k: v for k, v in layers.items() if "A" in v and "B" in v}
    return complete


# ---- Load all task LoRA weights ----
print("Loading LoRA weight databases...")
lora_weights_db = {}   # { task_name: { layer_key: {A, B} } }

for lora_dir in sorted(LORA_DIR.iterdir()):
    task_name = lora_dir.name.replace("loraretriever-llama2-7b-", "")
    try:
        weights = load_lora_ab(lora_dir)
        if weights:
            lora_weights_db[task_name] = weights
            print(f"  ✓ {task_name:40s}  layers={len(weights)}")
    except Exception as e:
        print(f"  ✗ {task_name}: {e}")

TASK_NAMES = sorted(lora_weights_db.keys())
print(f"\nLoaded {len(TASK_NAMES)} tasks.")

## 🎣 3. Activation Hook Infrastructure

To compute **activation-delta embeddings**, we need to intercept the hidden states `h_L` that flow *into* each LoRA-adapted linear layer during a forward pass. We register forward hooks on the base model's attention projection layers.

```
       ┌─────────────┐
h_L ──►│  base W_q   │──► h_q               ← we hook the INPUT here
       └─────────────┘
h_L ──►│  LoRA A_L   │──► h_L @ A_L^T
       │  LoRA B_L   │──► (h_L @ A_L^T) @ B_L^T * scaling  =  Δh_L
       └─────────────┘
```

In [ ]:
class ActivationCapture:
    """
    Context manager that registers forward hooks to capture input activations
    at specific named modules in the base model.
    
    Usage:
        with ActivationCapture(base_model, layer_keys) as caps:
            model(**inputs)
        # caps.activations[layer_key] = list of tensors, one per forward call
    """
    def __init__(self, model, layer_keys: List[str]):
        self.model      = model
        self.layer_keys = set(layer_keys)
        self.activations: Dict[str, List[torch.Tensor]] = defaultdict(list)
        self._hooks     = []
    
    def _make_hook(self, key):
        def hook(module, inputs, output):
            # inputs[0]: shape (batch, seq_len, d_in)
            # Detach, move to CPU, keep float32
            h = inputs[0].detach().cpu().float()
            self.activations[key].append(h)
        return hook
    
    def __enter__(self):
        self.activations.clear()
        # Walk the model tree and register hooks on target modules
        for name, module in self.model.named_modules():
            if name in self.layer_keys:
                h = module.register_forward_hook(self._make_hook(name))
                self._hooks.append(h)
        return self
    
    def __exit__(self, *args):
        for h in self._hooks:
            h.remove()
        self._hooks.clear()


def get_module_by_name(model, name: str):
    """Navigate model tree by dot-separated name string."""
    parts = name.split(".")
    obj   = model
    for p in parts:
        obj = getattr(obj, p)
    return obj


# Quick sanity check — verify we can capture activations
test_input  = tokenizer("Hello, world!", return_tensors="pt").to(device)
test_layers = LORA_LAYER_KEYS[:4]   # just first 4 for the check

with ActivationCapture(base_model, test_layers) as caps:
    with torch.no_grad():
        _ = base_model(**test_input)

print("Captured activations:")
for key in test_layers:
    acts = caps.activations[key]
    print(f"  {key.split('layers.')[1]:30s}  → shape {acts[0].shape}")

## 📊 4. Compute Activation-Delta Embeddings per Layer

For each `(task, layer)` pair:
1. Run the N few-shot inputs through the base model with hooks active
2. For each captured hidden state `h_L` ∈ ℝ^(batch × seq × d_in), compute:
   ```
   Δh_L = (h_L @ A_L^T) @ B_L^T  *  scaling    ∈ ℝ^(batch × seq × d_out)
   ```
3. Mean-pool over **tokens** → ℝ^(batch × d_out)
4. Mean-pool over **examples** → ℝ^(d_out)  
5. L2-normalise → final layer embedding

Since d_out can be large (4096 for Llama-2), we also optionally reduce via PCA.

In [ ]:
def load_task_inputs(task_name: str, n: int = N_SAMPLES) -> List[str]:
    """
    Load raw input strings for a task from its dataset JSON.
    Tries several filename conventions used by LoraRetriever.
    """
    candidates = [
        DATA_DIR / f"{task_name}.json",
        DATA_DIR / f"train_{task_name}.json",
    ]
    # Also try fuzzy search
    for candidate in list(DATA_DIR.glob(f"*{task_name}*.json")):
        candidates.append(candidate)
    
    for path in candidates:
        if path.exists():
            with open(path) as f:
                data = json.load(f)
            samples = random.sample(data, min(n, len(data)))
            return [s.get("input", "") for s in samples]
    
    raise FileNotFoundError(f"No dataset for task '{task_name}'")


def tokenize_texts(texts: List[str], max_len: int = MAX_SEQ_LEN) -> dict:
    """Tokenise a list of strings into padded batches."""
    return tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_len
    )


@torch.no_grad()
def compute_activation_delta(
    h: torch.Tensor,        # (batch, seq, d_in)
    A: torch.Tensor,        # (r, d_in)
    B: torch.Tensor,        # (d_out, r)
    attention_mask: Optional[torch.Tensor] = None,   # (batch, seq)
    scaling: float = LORA_SCALING
) -> torch.Tensor:
    """
    Compute Δh = (h @ A^T) @ B^T * scaling
    then mean-pool over non-padding tokens.
    Returns: (batch, d_out)
    """
    h = h.to(A.device)              # (batch, seq, d_in)
    after_A = h @ A.T               # (batch, seq, r)
    delta   = after_A @ B.T * scaling  # (batch, seq, d_out)
    
    if attention_mask is not None:
        mask = attention_mask.to(delta.device).unsqueeze(-1).float()  # (batch, seq, 1)
        delta = (delta * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
    else:
        delta = delta.mean(dim=1)   # (batch, d_out)
    
    return delta  # (batch, d_out)


@torch.no_grad()
def embed_lora_layer(
    task_name: str,
    layer_key: str,
    A: torch.Tensor,
    B: torch.Tensor,
    batch_size: int = 4
) -> torch.Tensor:
    """
    Compute the activation-delta embedding for one (task, layer) pair.
    Returns: normalised vector of shape (d_out,)
    """
    texts = load_task_inputs(task_name)
    A, B  = A.cpu(), B.cpu()
    
    all_deltas = []
    
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i : i + batch_size]
        inputs      = tokenize_texts(batch_texts)
        inputs_dev  = {k: v.to(device) for k, v in inputs.items()}
        
        with ActivationCapture(base_model, [layer_key]) as caps:
            base_model(**inputs_dev)
        
        # caps.activations[layer_key] = list of one tensor per forward call
        h = caps.activations[layer_key][0].cpu()  # (batch, seq, d_in)
        mask = inputs["attention_mask"].cpu()
        
        delta = compute_activation_delta(h, A, B, mask)  # (batch, d_out)
        all_deltas.append(delta)
    
    # Aggregate across all examples
    all_deltas = torch.cat(all_deltas, dim=0)       # (N, d_out)
    mean_delta  = all_deltas.mean(dim=0)             # (d_out,)
    normed      = F.normalize(mean_delta, dim=0)     # unit vector
    return normed


print("Embedding functions defined.")

In [ ]:
# ============================================================
# BUILD THE LAYER-WISE INDEX
# layer_index[task][layer_key] = embedding tensor (d_out,)
# This is the main computation — takes O(N_tasks × N_layers × N_examples / batch)
#
# On an A100 with 20 tasks × 64 layers × 20 examples: ~10-15 minutes
# Results are cached to disk so you only do this once.
# ============================================================

LAYER_INDEX_PATH = INDEX_CACHE / "layer_wise_index.pkl"

if LAYER_INDEX_PATH.exists():
    print("Loading cached layer-wise index...")
    with open(LAYER_INDEX_PATH, "rb") as f:
        layer_index = pickle.load(f)
    print(f"Loaded index: {len(layer_index)} tasks")
else:
    print("Building layer-wise index from scratch...")
    layer_index = {}   # { task_name: { layer_key: tensor(d_out) } }
    
    for task_name in tqdm(TASK_NAMES, desc="Tasks"):
        layer_weights = lora_weights_db[task_name]
        task_embs     = {}
        
        # Check that task has a dataset file
        has_data = any(
            (DATA_DIR / f"{task_name}.json").exists()
        )
        if not has_data:
            print(f"  Skipping {task_name} (no dataset)")
            continue
        
        for layer_key in tqdm(LORA_LAYER_KEYS, desc=task_name, leave=False):
            if layer_key not in layer_weights:
                continue
            A = layer_weights[layer_key]["A"]   # (r, d_in)
            B = layer_weights[layer_key]["B"]   # (d_out, r)
            
            try:
                emb = embed_lora_layer(task_name, layer_key, A, B)
                task_embs[layer_key] = emb
            except Exception as e:
                print(f"  Error embedding {task_name}/{layer_key}: {e}")
        
        if task_embs:
            layer_index[task_name] = task_embs
    
    # Save to disk
    with open(LAYER_INDEX_PATH, "wb") as f:
        pickle.dump(layer_index, f)
    print(f"\nIndex saved. {len(layer_index)} tasks indexed.")


# Report index statistics
all_tasks_indexed   = list(layer_index.keys())
sample_task         = all_tasks_indexed[0]
sample_layer_embs   = layer_index[sample_task]
sample_emb_dim      = next(iter(sample_layer_embs.values())).shape[0]

print(f"\nIndex summary:")
print(f"  Tasks indexed       : {len(all_tasks_indexed)}")
print(f"  Layers per task     : {len(sample_layer_embs)}")
print(f"  Embedding dimension : {sample_emb_dim}")
print(f"  Total embeddings    : {len(all_tasks_indexed) * len(sample_layer_embs):,}")

## 🗜️ 5. (Optional) Dimensionality Reduction via PCA

The raw embeddings live in ℝ^4096, which makes cosine search fast but storage large. PCA to 256 or 512 dims retains >95% of variance and speeds up retrieval significantly.

In [ ]:
from sklearn.decomposition import PCA

PCA_DIM  = 256    # set None to skip PCA
PCA_PATH = INDEX_CACHE / f"pca_{PCA_DIM}.pkl"

if PCA_DIM is not None:
    if PCA_PATH.exists():
        with open(PCA_PATH, "rb") as f:
            pca_models = pickle.load(f)  # one PCA per layer key
        print(f"Loaded PCA models from {PCA_PATH}")
    else:
        print(f"Fitting PCA (→{PCA_DIM} dims) per layer...")
        pca_models = {}  # { layer_key: sklearn PCA }
        
        for layer_key in tqdm(LORA_LAYER_KEYS, desc="Fitting PCA"):
            # Gather all task embeddings for this layer
            embs = []
            for task in all_tasks_indexed:
                if layer_key in layer_index[task]:
                    embs.append(layer_index[task][layer_key].numpy())
            
            if len(embs) < 2:
                continue
            
            X = np.stack(embs)  # (N_tasks, d_out)
            n_components = min(PCA_DIM, X.shape[0] - 1, X.shape[1])
            pca = PCA(n_components=n_components, random_state=SEED)
            pca.fit(X)
            pca_models[layer_key] = pca
        
        with open(PCA_PATH, "wb") as f:
            pickle.dump(pca_models, f)
        
        # Show explained variance for first few layers
        for i, layer_key in enumerate(list(pca_models.keys())[:3]):
            ev = pca_models[layer_key].explained_variance_ratio_.sum()
            print(f"  {layer_key.split('layers.')[1]:35s} explained var: {ev:.3f}")
    
    # Project all index embeddings
    print("Projecting index embeddings to reduced space...")
    layer_index_pca = {}  # { task: { layer_key: reduced_emb (PCA_DIM,) } }
    for task in all_tasks_indexed:
        layer_index_pca[task] = {}
        for layer_key, emb in layer_index[task].items():
            if layer_key in pca_models:
                reduced = pca_models[layer_key].transform(emb.numpy().reshape(1, -1))[0]
                # Re-normalise after projection
                reduced = reduced / (np.linalg.norm(reduced) + 1e-9)
                layer_index_pca[task][layer_key] = reduced
    
    ACTIVE_INDEX = layer_index_pca
    print(f"Using PCA-reduced index (dim={PCA_DIM})")
else:
    # Use raw embeddings (convert to numpy for cosine search)
    ACTIVE_INDEX = {
        task: {lk: emb.numpy() for lk, emb in embs.items()}
        for task, embs in layer_index.items()
    }
    print("Using raw (no PCA) index")

## 🔍 6. Layer-Wise Retrieval

For a query input, we compute the **activation delta** the query would produce at each layer — but since we don't have a LoRA yet, we use the base model's hidden states directly as a proxy for what "ideal activations" look like at each layer.

Then for each layer independently, we find the task whose activation-delta embedding is most similar.

**Query embedding at layer L = the base model's hidden state at layer L, mean-pooled over tokens, L2-normalised.**  
This tells us: *what kind of representation is flowing through this layer for this specific input?*

In [ ]:
@torch.no_grad()
def compute_query_layer_embeddings(
    query_text: str,
    layer_keys: List[str]
) -> Dict[str, np.ndarray]:
    """
    For a query string, compute an embedding at each layer by capturing
    the base model hidden states and mean-pooling over tokens.
    
    If PCA is enabled, we project the query embeddings too.
    Returns: { layer_key: np.ndarray (d or PCA_DIM,) }
    """
    inputs     = tokenize_texts([query_text])
    inputs_dev = {k: v.to(device) for k, v in inputs.items()}
    mask       = inputs["attention_mask"].float()  # (1, seq)
    
    # Single forward pass — capture ALL layer inputs simultaneously
    with ActivationCapture(base_model, layer_keys) as caps:
        base_model(**inputs_dev)
    
    query_embs = {}
    for layer_key in layer_keys:
        if layer_key not in caps.activations:
            continue
        h    = caps.activations[layer_key][0].cpu()  # (1, seq, d)
        # Masked mean-pool over tokens
        m    = mask.unsqueeze(-1)                     # (1, seq, 1)
        pooled = (h * m).sum(dim=1) / m.sum(dim=1).clamp(min=1)  # (1, d)
        normed = F.normalize(pooled[0], dim=0).numpy()            # (d,)
        
        # Optional PCA projection
        if PCA_DIM is not None and layer_key in pca_models:
            normed = pca_models[layer_key].transform(normed.reshape(1, -1))[0]
            normed = normed / (np.linalg.norm(normed) + 1e-9)
        
        query_embs[layer_key] = normed
    
    return query_embs


def retrieve_layer_wise(
    query_text: str,
    top_k_per_layer: int = 1,
    layer_keys: Optional[List[str]] = None
) -> Dict[str, List[Tuple[str, float]]]:
    """
    For each layer, independently retrieve the top-k tasks.
    
    Returns:
        { layer_key: [(task_name, similarity_score), ...] }
        sorted by descending similarity per layer.
    """
    if layer_keys is None:
        layer_keys = LORA_LAYER_KEYS
    
    # Step 1: Get query embeddings for all layers in ONE forward pass
    query_embs = compute_query_layer_embeddings(query_text, layer_keys)
    
    # Step 2: For each layer, cosine similarity against all task embeddings
    all_tasks = list(ACTIVE_INDEX.keys())
    results   = {}
    
    for layer_key in layer_keys:
        if layer_key not in query_embs:
            continue
        
        q = query_embs[layer_key]  # (d,) or (PCA_DIM,)
        
        scores = []
        for task in all_tasks:
            if layer_key not in ACTIVE_INDEX[task]:
                continue
            t    = ACTIVE_INDEX[task][layer_key]  # (d,)
            sim  = float(np.dot(q, t))            # cosine sim (both normalised)
            scores.append((task, sim))
        
        scores.sort(key=lambda x: -x[1])
        results[layer_key] = scores[:top_k_per_layer]
    
    return results


# ---- Test retrieval ----
test_query = "Classify the sentiment: The movie was absolutely fantastic!"
print(f"Query: {test_query}\n")

layer_assignments = retrieve_layer_wise(test_query, top_k_per_layer=1)

print(f"{'Layer':45s} {'Best Task':30s} {'Sim':>6}")
print("-" * 85)
for layer_key in LORA_LAYER_KEYS[:16]:   # show first 16 layers
    if layer_key in layer_assignments:
        best_task, best_sim = layer_assignments[layer_key][0]
        short_key = layer_key.replace("model.layers.", "L").replace(".self_attn.", ".")
        print(f"  {short_key:43s} {best_task:30s} {best_sim:6.4f}")

In [ ]:
# ---- Visualise: which task dominates at each layer? ----
# A heatmap showing task assignment patterns across transformer depth

def visualise_layer_assignments(
    query_text: str,
    top_k: int = 3,
    module_to_show: str = "q_proj"
):
    """Show which tasks are retrieved at each transformer layer depth."""
    assignments = retrieve_layer_wise(query_text, top_k_per_layer=top_k)
    
    # Filter to one module type for clarity
    filtered_keys = [
        k for k in LORA_LAYER_KEYS if module_to_show in k
    ]
    
    task_set   = sorted(all_tasks_indexed)
    task_to_id = {t: i for i, t in enumerate(task_set)}
    
    # Build assignment matrix (layer × top_k)
    data = []
    layer_labels = []
    for lk in filtered_keys:
        if lk not in assignments:
            continue
        layer_num = int(lk.split("layers.")[1].split(".")[0])
        layer_labels.append(f"L{layer_num}")
        row = [task_to_id[t] for t, _ in assignments[lk]]
        data.append(row)
    
    data = np.array(data)  # (n_layers, top_k)
    
    fig, ax = plt.subplots(figsize=(max(4, top_k * 2), max(8, len(layer_labels) * 0.35)))
    im = ax.imshow(data, aspect='auto', cmap='tab20', vmin=0, vmax=len(task_set)-1)
    ax.set_yticks(range(len(layer_labels)))
    ax.set_yticklabels(layer_labels, fontsize=8)
    ax.set_xticks(range(top_k))
    ax.set_xticklabels([f"Rank {i+1}" for i in range(top_k)])
    ax.set_title(f"Layer-wise LoRA Retrieval ({module_to_show})\nQuery: {query_text[:60]}...", fontsize=10)
    
    # Add task name annotations inside cells
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            task_short = task_set[data[i, j]][:12]
            ax.text(j, i, task_short, ha='center', va='center', fontsize=5, color='white',
                    fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(INDEX_CACHE / "layer_assignments.png", dpi=150, bbox_inches='tight')
    plt.show()
    return assignments


queries_to_visualise = [
    "Classify the sentiment: The movie was absolutely fantastic!",
    "Does the hypothesis follow from the premise? Premise: All birds can fly. Hypothesis: Penguins can fly.",
    "Summarize the following passage in one sentence.",
]

for q in queries_to_visualise:
    print(f"\nQuery: {q[:70]}")
    visualise_layer_assignments(q, top_k=3, module_to_show="q_proj")

## 🔨 7. Build a Hybrid LoRA from Per-Layer Selections

After retrieval we have, for each LoRA layer, the best-matching task. We now materialise a **single valid PEFT adapter** whose weights are taken from different tasks layer-by-layer.

```
Hybrid LoRA:
  L0  q_proj  →  A, B  from  task_sst2
  L0  v_proj  →  A, B  from  task_mnli
  L1  q_proj  →  A, B  from  task_squad
  ...etc
```

This is a drop-in replacement for any normal LoRA — it can be loaded with `PeftModel.from_pretrained` using a patched adapter config.

In [ ]:
import safetensors.torch as sf
import shutil

def build_hybrid_lora(
    layer_assignments: Dict[str, List[Tuple[str, float]]],
    output_dir: Path,
    weights_strategy: str = "best",   # "best" | "weighted" | "soft_merge"
    top_k_blend: int = 1
) -> Path:
    """
    Materialise a hybrid LoRA adapter by assembling per-layer weights
    from different tasks according to layer_assignments.
    
    Args:
        layer_assignments : output of retrieve_layer_wise()
        output_dir        : where to save the hybrid adapter
        weights_strategy  : 
            "best"        → use the top-1 task's A, B directly
            "weighted"    → similarity-weighted average of top-k A·B  (fused ΔW, re-factored)
            "soft_merge"  → same as weighted but scales by sim softmax with temperature T
        top_k_blend       : number of tasks to blend (only for weighted/soft_merge)
    
    Returns: path to the saved hybrid adapter directory
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Copy adapter_config.json from any reference LoRA (all share the same arch)
    ref_lora_dir = next(LORA_DIR.iterdir())
    shutil.copy(ref_lora_dir / "adapter_config.json", output_dir / "adapter_config.json")
    
    hybrid_state_dict = {}
    provenance        = {}   # track which task supplied each layer (for analysis)
    
    for layer_key in LORA_LAYER_KEYS:
        if layer_key not in layer_assignments:
            # Fall back to reference LoRA if no assignment exists
            ref_task = TASK_NAMES[0]
            if layer_key in lora_weights_db[ref_task]:
                layer_assignments[layer_key] = [(ref_task, 0.0)]
            else:
                continue
        
        candidates = layer_assignments[layer_key]  # [(task, sim), ...]
        
        if weights_strategy == "best" or len(candidates) == 1:
            # ── Strategy: Top-1 selection ──────────────────────────────────
            best_task = candidates[0][0]
            if layer_key not in lora_weights_db.get(best_task, {}):
                continue
            A = lora_weights_db[best_task][layer_key]["A"]  # (r, d_in)
            B = lora_weights_db[best_task][layer_key]["B"]  # (d_out, r)
            provenance[layer_key] = best_task
        
        else:
            # ── Strategy: Weighted blend of top-k ΔW matrices ──────────────
            # We compute fused_ΔW = Σᵢ wᵢ (Aᵢ Bᵢ) then re-factorise via SVD
            # to get new A' and B' of the same rank r.
            k = min(top_k_blend, len(candidates))
            sims = np.array([s for _, s in candidates[:k]])
            
            if weights_strategy == "soft_merge":
                T = 0.05  # temperature
                ws = np.exp(sims / T); ws /= ws.sum()
            else:
                ws = sims / sims.sum()  # raw proportional
            
            fused_dW = None
            for i, (task, _) in enumerate(candidates[:k]):
                if layer_key not in lora_weights_db.get(task, {}):
                    continue
                Ai = lora_weights_db[task][layer_key]["A"].float()  # (r, d_in)
                Bi = lora_weights_db[task][layer_key]["B"].float()  # (d_out, r)
                dWi = ws[i] * (Bi @ Ai)                              # (d_out, d_in)
                fused_dW = dWi if fused_dW is None else fused_dW + dWi
            
            if fused_dW is None:
                continue
            
            # Re-factorise fused_ΔW back into low-rank form via truncated SVD
            # fused_dW ∈ ℝ^{d_out × d_in}  →  U S V^T,  keep rank r
            U, S, Vh = torch.linalg.svd(fused_dW, full_matrices=False)
            r = LORA_R
            # B' = U[:, :r] * sqrt(S[:r])    shape: (d_out, r)
            # A' = diag(sqrt(S[:r])) @ Vh[:r, :]  shape: (r, d_in)
            sqrt_S = torch.sqrt(S[:r])                  # (r,)
            B      = U[:, :r] * sqrt_S.unsqueeze(0)     # (d_out, r)
            A      = Vh[:r, :] * sqrt_S.unsqueeze(1)    # (r, d_in)
            # Rescale by 1/scaling so the forward pass gives fused_dW unchanged
            B = B / LORA_SCALING
            
            provenance[layer_key] = "+".join(t for t, _ in candidates[:k])
        
        # Store in PEFT state dict format
        # Key format: "base_model.model.{layer_key}.lora_A.weight"
        prefix = f"base_model.model.{layer_key}"
        hybrid_state_dict[f"{prefix}.lora_A.weight"] = A.half()   # save as fp16
        hybrid_state_dict[f"{prefix}.lora_B.weight"] = B.half()
    
    # Save the hybrid adapter weights
    sf.save_file(hybrid_state_dict, str(output_dir / "adapter_model.safetensors"))
    
    # Save provenance (which task provided each layer)
    with open(output_dir / "provenance.json", "w") as f:
        json.dump(provenance, f, indent=2)
    
    print(f"Hybrid LoRA saved to {output_dir}")
    print(f"  Layers assembled: {len(hybrid_state_dict) // 2}")
    
    # Show diversity: how many distinct tasks contributed?
    contributing_tasks = set(
        t.split("+")[0] for t in provenance.values()
    )
    print(f"  Contributing tasks: {len(contributing_tasks)}  → {sorted(contributing_tasks)}")
    
    return output_dir


print("build_hybrid_lora() defined.")

## 🤖 8. End-to-End Inference with Layer-Wise Hybrid LoRA

In [ ]:
ALPACA_TEMPLATE = (
    "Below is an instruction that describes a task, paired with an input that provides "
    "further context. Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n### Input:\n{input}\n\n### Response:"
)
ALPACA_NO_INPUT = (
    "Below is an instruction that describes a task. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n### Response:"
)

def format_prompt(instruction, input_text=""):
    if input_text.strip():
        return ALPACA_TEMPLATE.format(instruction=instruction, input=input_text)
    return ALPACA_NO_INPUT.format(instruction=instruction)


@torch.no_grad()
def generate(model, prompt, max_new_tokens=128):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outs   = model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        do_sample=False, pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(outs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def layerwise_loraretriever(
    instruction: str,
    input_text:  str = "",
    strategy:    str = "best",   # "best" | "weighted" | "soft_merge"
    top_k:       int = 3,
    verbose:     bool = True
) -> dict:
    """
    Full layer-wise LoraRetriever pipeline.
    
    1. Retrieve: for each LoRA layer, find the best-matching task(s)
    2. Build hybrid LoRA from per-layer selections
    3. Load as a PEFT model and generate
    """
    query = f"{instruction} {input_text}".strip()
    
    if verbose:
        print(f"\n{'='*65}")
        print(f"Query: {query[:80]}")
    
    # Step 1: Layer-wise retrieval
    layer_assignments = retrieve_layer_wise(query, top_k_per_layer=top_k)
    
    if verbose:
        # Summarise — show the distribution of top-1 tasks across layers
        task_counts = defaultdict(int)
        for lk in layer_assignments:
            task_counts[layer_assignments[lk][0][0]] += 1
        print("Layer assignment distribution (top-1):")
        for task, cnt in sorted(task_counts.items(), key=lambda x: -x[1]):
            bar = '█' * cnt
            print(f"  {task:35s} {bar} ({cnt})")
    
    # Step 2: Build hybrid LoRA
    hybrid_dir = INDEX_CACHE / "hybrid_lora_tmp"
    build_hybrid_lora(
        layer_assignments=layer_assignments,
        output_dir=hybrid_dir,
        weights_strategy=strategy,
        top_k_blend=top_k
    )
    
    # Step 3: Load hybrid LoRA and generate
    model  = PeftModel.from_pretrained(base_model, str(hybrid_dir))
    model.eval()
    
    prompt   = format_prompt(instruction, input_text)
    response = generate(model, prompt)
    
    if verbose:
        print(f"Response: {response}")
    
    # Clean up
    del model
    torch.cuda.empty_cache()
    
    return {
        "query": query,
        "layer_assignments": layer_assignments,
        "strategy": strategy,
        "response": response
    }


# ---- Run on diverse examples ----
examples = [
    {"instruction": "Classify the sentiment of the review.",
     "input": "Absolutely loved the performances and cinematography!"},
    {"instruction": "Does the hypothesis logically follow from the premise?",
     "input": "Premise: All mammals are warm-blooded. Hypothesis: Whales are warm-blooded."},
    {"instruction": "Answer the question based on the context.",
     "input": "Context: The Great Wall was built over many centuries. Question: Was the Great Wall built all at once?"},
]

for ex in examples:
    result = layerwise_loraretriever(
        instruction=ex["instruction"],
        input_text=ex["input"],
        strategy="soft_merge",
        top_k=3
    )

## 📊 9. Ablation: Original vs Layer-Wise Retrieval

In [ ]:
# ---- Rebuild original task-level index for comparison ----
# The original method uses instructor-xl mean-pooled input embeddings
# We mirror that here directly in PyTorch to avoid needing the INSTRUCTOR library

from sentence_transformers import SentenceTransformer

RETRIEVER = SentenceTransformer("hkunlp/instructor-xl")
RETRIEVER.eval()
INSTR_PREFIX = "Represent the sentence for similar task retrieval:"

@torch.no_grad()
def embed_texts_instructor(texts, instruction=INSTR_PREFIX):
    pairs = [[instruction, t] for t in texts]
    return RETRIEVER.encode(pairs, batch_size=8, show_progress_bar=False)

# Build task-level index
task_index = {}  # { task_name: np.ndarray (768,) }
for task_name in TASK_NAMES:
    data_file = DATA_DIR / f"{task_name}.json"
    if not data_file.exists():
        continue
    with open(data_file) as f:
        data = json.load(f)
    texts = [d["input"] for d in random.sample(data, min(20, len(data)))]
    embs  = embed_texts_instructor(texts)
    task_index[task_name] = embs.mean(axis=0)

task_keys   = list(task_index.keys())
task_matrix = np.stack([task_index[k] for k in task_keys])
task_matrix = task_matrix / np.linalg.norm(task_matrix, axis=1, keepdims=True)
print(f"Task-level index: {task_matrix.shape}")

In [ ]:
from rouge_score import rouge_scorer as rouge_lib
import re

def norm(s):
    s = s.lower().strip()
    s = re.sub(r'[^\w\s]', '', s)
    return ' '.join(s.split())

def exact_match(pred, ref):
    return int(norm(pred) == norm(ref))

def rouge_l(pred, ref):
    scorer = rouge_lib.RougeScorer(['rougeL'], use_stemmer=True)
    return scorer.score(ref, pred)['rougeL'].fmeasure


def run_eval_comparison(eval_task_names, n_examples=50):
    """
    Compare three approaches on the same eval set:
      A) Original LoraRetriever (task-level retrieval, parameter fusion)
      B) Layer-wise retrieval (Top-1 per layer, "best" strategy)
      C) Layer-wise retrieval (weighted SVD blend, "soft_merge" strategy)
    """
    records = []
    
    for task_name in eval_task_names:
        data_file = DATA_DIR / f"{task_name}.json"
        if not data_file.exists():
            continue
        with open(data_file) as f:
            all_data = json.load(f)
        
        eval_data = all_data[:n_examples]
        
        for method in ["original_task", "layerwise_best", "layerwise_soft"]:
            preds, refs = [], []
            
            for ex in tqdm(eval_data, desc=f"{task_name}/{method}", leave=False):
                instruction = ex.get("instruction", "")
                input_text  = ex.get("input", "")
                label       = ex.get("output", "")
                query       = f"{instruction} {input_text}".strip()
                
                if method == "original_task":
                    # Task-level retrieval + parameter fusion
                    q_emb = embed_texts_instructor([query])[0]
                    q_emb = q_emb / np.linalg.norm(q_emb)
                    sims  = task_matrix @ q_emb
                    top3  = np.argsort(sims)[::-1][:3]
                    # Parameter fusion (weighted mean of ΔW)
                    ws    = sims[top3]; ws = np.exp(ws/0.1); ws /= ws.sum()
                    
                    # Build fused state dict
                    fused = {}
                    for i, idx in enumerate(top3):
                        t_name = task_keys[idx]
                        for lk, ab in lora_weights_db.get(t_name, {}).items():
                            dW = ws[i] * LORA_SCALING * (ab["B"].float() @ ab["A"].float())
                            fused[lk] = fused.get(lk, 0) + dW
                    
                    # Apply to base model (temporary patching)
                    model = copy.deepcopy(base_model)
                    with torch.no_grad():
                        for name, param in model.named_parameters():
                            clean = name.replace(".weight", "")
                            if clean in fused:
                                param.data += fused[clean].to(param.device, param.dtype)
                    
                elif method in ("layerwise_best", "layerwise_soft"):
                    strat = "best" if method == "layerwise_best" else "soft_merge"
                    l_assigns = retrieve_layer_wise(query, top_k_per_layer=3)
                    hybrid_dir = INDEX_CACHE / f"eval_hybrid_{method}"
                    build_hybrid_lora(l_assigns, hybrid_dir, weights_strategy=strat, top_k_blend=3)
                    model = PeftModel.from_pretrained(base_model, str(hybrid_dir))
                    model.eval()
                
                prompt = format_prompt(instruction, input_text)
                pred   = generate(model, prompt)
                preds.append(pred)
                refs.append(label)
                
                del model
                torch.cuda.empty_cache()
            
            em   = np.mean([exact_match(p, r) for p, r in zip(preds, refs)])
            rl   = np.mean([rouge_l(p, r)     for p, r in zip(preds, refs)])
            records.append({"task": task_name, "method": method, "EM": em, "RougeL": rl})
            print(f"  {task_name:25s} {method:20s} EM={em:.3f}  RougeL={rl:.3f}")
    
    return pd.DataFrame(records)


# Evaluate on 3 tasks (expand to all for full reproduction)
eval_tasks = TASK_NAMES[:3]
results_df = run_eval_comparison(eval_tasks, n_examples=50)
print("\n", results_df.pivot_table(index="task", columns="method", values=["EM", "RougeL"]).round(3))

In [ ]:
# ---- Bar chart comparison ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric in zip(axes, ["EM", "RougeL"]):
    pivot = results_df.pivot(index="task", columns="method", values=metric)
    pivot.plot(kind="bar", ax=ax, colormap="Set2", edgecolor="black", linewidth=0.5)
    ax.set_title(metric)
    ax.set_xlabel("Task")
    ax.set_ylabel("Score")
    ax.tick_params(axis='x', rotation=30)
    ax.legend(fontsize=8, loc="upper right")
    ax.set_ylim(0, 1)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle("Original Task-Level vs Layer-Wise LoRA Retrieval", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(INDEX_CACHE / "ablation_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

## 🔬 10. Analysis: Layer Specialisation Patterns

A key question: **do different layers actually favour different tasks?** Or does every layer pick the same winner?

If the layer-wise approach adds value, we'd expect to see:
- Early layers (0–10) tending to retrieve structurally-similar tasks
- Later layers (20–31) tending to retrieve semantically/task-similar ones
- For a multi-skill query, the assignments should genuinely split across tasks

In [ ]:
def analyse_layer_specialisation(n_queries: int = 20):
    """
    Run N queries and measure:
    1. Per-layer task-assignment entropy (how diverse are the assignments per layer?)
    2. Cross-layer agreement (do layers agree with each other?)
    3. Early vs late layer task preferences
    """
    all_assignments = []  # list of {layer_key: task} per query
    
    # Sample queries from different tasks
    for task_name in random.sample(TASK_NAMES, min(n_queries, len(TASK_NAMES))):
        data_file = DATA_DIR / f"{task_name}.json"
        if not data_file.exists():
            continue
        with open(data_file) as f:
            data = json.load(f)
        sample    = random.choice(data)
        query     = sample["input"]
        assigns   = retrieve_layer_wise(query, top_k_per_layer=1)
        flat      = {lk: assigns[lk][0][0] for lk in assigns}
        all_assignments.append({"query_task": task_name, **flat})
    
    df = pd.DataFrame(all_assignments)
    
    # --- Plot 1: Per-layer assignment entropy ---
    from scipy.stats import entropy
    
    layer_entropies = {}
    for lk in LORA_LAYER_KEYS:
        if lk not in df.columns:
            continue
        counts = df[lk].value_counts(normalize=True).values
        layer_entropies[lk] = float(entropy(counts))
    
    # Plot entropy by layer index and module
    fig, axes = plt.subplots(1, len(TARGET_MODULES), figsize=(5 * len(TARGET_MODULES), 4), sharey=True)
    if len(TARGET_MODULES) == 1:
        axes = [axes]
    
    for ax, mod in zip(axes, TARGET_MODULES):
        xs, ys = [], []
        for L in range(N_LAYERS):
            lk = f"model.layers.{L}.self_attn.{mod}"
            if lk in layer_entropies:
                xs.append(L)
                ys.append(layer_entropies[lk])
        ax.plot(xs, ys, marker='o', markersize=4, linewidth=1.5)
        ax.set_title(f"{mod}")
        ax.set_xlabel("Transformer layer index")
        ax.set_ylabel("Assignment entropy" if mod == TARGET_MODULES[0] else "")
        ax.grid(alpha=0.3)
    
    plt.suptitle("Per-Layer Task Assignment Entropy\n(higher = more diverse task selection)",
                 fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig(INDEX_CACHE / "layer_entropy.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    # --- Summary ---
    early_mean = np.mean([layer_entropies.get(f"model.layers.{L}.self_attn.{TARGET_MODULES[0]}", 0)
                          for L in range(N_LAYERS // 3)])
    late_mean  = np.mean([layer_entropies.get(f"model.layers.{L}.self_attn.{TARGET_MODULES[0]}", 0)
                          for L in range(2 * N_LAYERS // 3, N_LAYERS)])
    print(f"\nMean assignment entropy:")
    print(f"  Early layers (0–{N_LAYERS//3}):          {early_mean:.3f}")
    print(f"  Late  layers ({2*N_LAYERS//3}–{N_LAYERS}): {late_mean:.3f}")
    print(f"  (Higher entropy = more diverse task assignments at that layer)")
    
    return df, layer_entropies


spec_df, entropies = analyse_layer_specialisation(n_queries=20)

## 🗺️ 11. Layer Embedding Space Visualisation (t-SNE / UMAP)

Visualise whether different tasks cluster differently at different layers — validating that layer-specific embeddings carry distinct information.

In [ ]:
from sklearn.manifold import TSNE

def visualise_layer_embedding_space(layer_key: str):
    """
    t-SNE of all task embeddings at a specific layer.
    Each point is a task; colour = task cluster (if metadata available).
    """
    tasks, embs = [], []
    for task in all_tasks_indexed:
        if layer_key in ACTIVE_INDEX[task]:
            tasks.append(task)
            embs.append(ACTIVE_INDEX[task][layer_key])
    
    if len(embs) < 4:
        print(f"Too few tasks for {layer_key}")
        return
    
    X      = np.stack(embs)                  # (N, d)
    perp   = min(5, len(tasks) - 1)
    coords = TSNE(n_components=2, perplexity=perp, random_state=SEED).fit_transform(X)
    
    plt.figure(figsize=(8, 6))
    plt.scatter(coords[:, 0], coords[:, 1], s=60, alpha=0.8, c=range(len(tasks)), cmap='tab20')
    for i, t in enumerate(tasks):
        plt.annotate(t[:12], (coords[i, 0], coords[i, 1]), fontsize=6, alpha=0.75,
                     xytext=(3, 3), textcoords='offset points')
    
    layer_short = layer_key.replace("model.layers.", "L").replace(".self_attn.", ".")
    plt.title(f"t-SNE of LoRA layer embeddings — {layer_short}", fontsize=11)
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(INDEX_CACHE / f"tsne_{layer_short.replace('.','-')}.png", dpi=150)
    plt.show()


# Compare embedding space at early vs late layers
for L in [0, N_LAYERS // 2, N_LAYERS - 1]:
    layer_key = f"model.layers.{L}.self_attn.{TARGET_MODULES[0]}"
    visualise_layer_embedding_space(layer_key)

## 💾 12. Save Everything for Reuse

In [ ]:
# Save the complete layer-wise index with metadata
save_bundle = {
    "layer_index":      layer_index,          # raw fp32 embeddings
    "active_index":     ACTIVE_INDEX,         # possibly PCA-reduced
    "task_names":       TASK_NAMES,
    "lora_layer_keys":  LORA_LAYER_KEYS,
    "target_modules":   TARGET_MODULES,
    "n_layers":         N_LAYERS,
    "pca_dim":          PCA_DIM,
    "model_id":         BASE_MODEL_ID,
    "lora_r":           LORA_R,
    "lora_scaling":     LORA_SCALING,
}

with open(INDEX_CACHE / "full_bundle.pkl", "wb") as f:
    pickle.dump(save_bundle, f)

if PCA_DIM:
    with open(PCA_PATH, "wb") as f:
        pickle.dump(pca_models, f)

print("All index files saved to", INDEX_CACHE)
print("\nTo reload:")
print("  with open('./layer_wise_index/full_bundle.pkl', 'rb') as f:")
print("      bundle = pickle.load(f)")
print("  ACTIVE_INDEX = bundle['active_index']")

---

## 📋 Summary: What This Notebook Does

| Step | What | Key Design Decision |
|------|------|---------------------|
| Load weights | Parse A, B matrices per layer per task | Canonical `model.layers.L.self_attn.mod` key format |
| Embed layers | `Δh = h @ A^T @ B^T * scale`, mean-pool over tokens + examples | Input-distribution-aware, not just weight geometry |
| PCA (optional) | Reduce 4096 → 256 dims per layer | Fit one PCA per layer key |
| Query embed | Single forward pass captures ALL layers at once | O(1) forward pass per query, not O(N_layers) |
| Retrieval | Per-layer cosine similarity; returns `{layer_key: [(task, sim)]}` | Each layer independently picks its best task |
| Hybrid build | Assemble `adapter_model.safetensors` from per-layer task winners | SVD re-factorisation for blended ΔW |
| Inference | Load hybrid as normal PEFT model | No custom inference code needed |
| Analysis | Entropy per layer, t-SNE, ablation comparison | Validates that layers are genuinely specialised |

### Three Composition Strategies Compared

| Strategy | What it does | When to use |
|----------|-------------|-------------|
| `best` | Top-1 task's A/B directly for each layer | Fastest; best when one task dominates |
| `weighted` | Weighted mean of top-k ΔW, re-factored via SVD | Balanced blend; raw similarity weights |
| `soft_merge` | Same but weights via temperature-scaled softmax | Sharper: strongly prefers the top retrieval hit |